In [1]:
# # ============================================================
# # FEW-SHOT FOOD CLASSIFICATION (80 CLASSES)
# # CLIP + Prototypical Networks (UPGRADED VERSION)
# # ============================================================

# import os
# import random
# import torch
# import torch.nn.functional as F
# from torchvision import transforms
# from PIL import Image
# from collections import defaultdict
# from transformers import CLIPVisionModel

# # ======================
# # CONFIGURATION
# # ======================
# TRAIN_DATASET_ROOT = "/kaggle/input/cv-dataset/Project Data/Food/Train"
# TEST_DATASET_ROOT  = "/kaggle/input/cv-dataset/Project Data/Food/Validation"

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# N_WAY = 10
# K_SHOT = 5
# Q_QUERY = 5
# EPISODES_PER_EPOCH = 200
# EPOCHS = 25
# IMG_SIZE = 224

# # ======================
# # BUILD CLASS INDEX
# # ======================
# def build_class_index(root):
#     class_to_images = defaultdict(list)
#     for cls in os.listdir(root):
#         cls_path = os.path.join(root, cls)
#         if not os.path.isdir(cls_path):
#             continue
#         for img in os.listdir(cls_path):
#             if img.lower().endswith((".jpg", ".jpeg", ".png")):
#                 class_to_images[cls].append(os.path.join(cls_path, img))
#     return class_to_images

# train_data = build_class_index(TRAIN_DATASET_ROOT)
# test_data  = build_class_index(TEST_DATASET_ROOT)

# # ======================
# # TRANSFORMS (CLIP CORRECT)
# # ======================
# CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
# CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

# train_tf = transforms.Compose([
#     transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomApply([
#         transforms.ColorJitter(0.5, 0.5, 0.5, 0.2)
#     ], p=0.8),
#     transforms.RandomGrayscale(p=0.1),
#     transforms.GaussianBlur(3, sigma=(0.1, 2.0)),
#     transforms.ToTensor(),
#     transforms.Normalize(CLIP_MEAN, CLIP_STD),
# ])

# test_tf = transforms.Compose([
#     transforms.Resize(int(IMG_SIZE * 1.1)),
#     transforms.CenterCrop(IMG_SIZE),
#     transforms.ToTensor(),
#     transforms.Normalize(CLIP_MEAN, CLIP_STD),
# ])

# # ======================
# # EPISODE SAMPLER
# # ======================
# def sample_k(images, k):
#     if len(images) >= k:
#         return random.sample(images, k)
#     else:
#         return random.choices(images, k=k)

# def sample_episode(data, n_way, k_shot, q_query):
#     selected = random.sample(list(data.keys()), n_way)
#     support_x, support_y, query_x, query_y = [], [], [], []

#     for label, cls in enumerate(selected):
#         imgs = sample_k(data[cls], k_shot + q_query)
#         support_x += imgs[:k_shot]
#         query_x += imgs[k_shot:]
#         support_y += [label] * k_shot
#         query_y += [label] * q_query

#     return support_x, support_y, query_x, query_y

# # ======================
# # IMAGE LOADER
# # ======================
# def load_images(paths, transform):
#     images = []
#     for p in paths:
#         img = Image.open(p).convert("RGB")
#         images.append(transform(img))
#     return torch.stack(images)

# # ======================
# # CLIP MODEL
# # ======================
# clip_model = CLIPVisionModel.from_pretrained(
#     "openai/clip-vit-base-patch32"
# ).to(DEVICE)

# # Learnable temperature
# temperature = torch.nn.Parameter(torch.tensor(10.0).to(DEVICE))

# optimizer = torch.optim.AdamW(
#     list(clip_model.parameters()) + [temperature],
#     lr=3e-5,
#     weight_decay=1e-4
# )

# # ======================
# # FREEZE / UNFREEZE UTILS
# # ======================
# def freeze_all(model):
#     for p in model.parameters():
#         p.requires_grad = False

# def unfreeze_last_blocks(model, n=4):
#     for block in model.vision_model.encoder.layers[-n:]:
#         for p in block.parameters():
#             p.requires_grad = True

# # ======================
# # TRAINING LOOP
# # ======================
# for epoch in range(EPOCHS):

#     # Freeze strategy
#     if epoch < 5:
#         freeze_all(clip_model)
#     elif epoch < 15:
#         freeze_all(clip_model)
#         unfreeze_last_blocks(clip_model, 4)
#     else:
#         for p in clip_model.parameters():
#             p.requires_grad = True

#     clip_model.train()
#     total_loss = 0

#     for _ in range(EPISODES_PER_EPOCH):
#         sx_paths, sy, qx_paths, qy = sample_episode(
#             train_data, N_WAY, K_SHOT, Q_QUERY
#         )

#         sx = load_images(sx_paths, train_tf).to(DEVICE)
#         qx = load_images(qx_paths, train_tf).to(DEVICE)
#         qy = torch.tensor(qy).to(DEVICE)
#         sy = torch.tensor(sy).to(DEVICE)

#         # CLIP embeddings (CLS token)
#         s_emb = clip_model(pixel_values=sx).last_hidden_state[:, 0, :]
#         q_emb = clip_model(pixel_values=qx).last_hidden_state[:, 0, :]

#         s_emb = F.normalize(s_emb, dim=-1)
#         q_emb = F.normalize(q_emb, dim=-1)

#         # Prototypes
#         prototypes = torch.stack([
#             s_emb[sy == c].mean(0) for c in range(N_WAY)
#         ])
#         prototypes = F.normalize(prototypes, dim=-1)

#         # Cosine similarity + temperature
#         logits = F.cosine_similarity(
#             q_emb.unsqueeze(1),
#             prototypes.unsqueeze(0),
#             dim=-1
#         ) * temperature

#         loss = F.cross_entropy(logits, qy)

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/EPISODES_PER_EPOCH:.4f}")

# # ======================
# # EVALUATION
# # ======================
# clip_model.eval()
# correct, total = 0, 0

# with torch.no_grad():
#     for _ in range(1000):
#         sx_paths, sy, qx_paths, qy = sample_episode(
#             test_data, N_WAY, K_SHOT, Q_QUERY
#         )

#         sx = load_images(sx_paths, test_tf).to(DEVICE)
#         qx = load_images(qx_paths, test_tf).to(DEVICE)
#         qy = torch.tensor(qy).to(DEVICE)
#         sy = torch.tensor(sy).to(DEVICE)

#         s_emb = clip_model(pixel_values=sx).last_hidden_state[:, 0, :]
#         q_emb = clip_model(pixel_values=qx).last_hidden_state[:, 0, :]

#         s_emb = F.normalize(s_emb, dim=-1)
#         q_emb = F.normalize(q_emb, dim=-1)

#         prototypes = torch.stack([
#             s_emb[sy == c].mean(0) for c in range(N_WAY)
#         ])
#         prototypes = F.normalize(prototypes, dim=-1)

#         preds = torch.argmax(
#             F.cosine_similarity(
#                 q_emb.unsqueeze(1),
#                 prototypes.unsqueeze(0),
#                 dim=-1
#             ),
#             dim=1
#         )

#         correct += (preds == qy).sum().item()
#         total += qy.size(0)

# print(f"\n🔥 Few-Shot Accuracy: {100 * correct / total:.2f}%")


In [2]:
# # ============================================================
# # HEALTHY & STRONG SIAMESE NETWORK (CLIP BACKBONE)
# # Triplet Loss + Semi-Hard Mining
# # ============================================================

# import os, random
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torchvision import transforms
# from transformers import CLIPVisionModel
# from PIL import Image
# from collections import defaultdict
# from torch.cuda.amp import autocast, GradScaler

# # ======================
# # CONFIG
# # ======================
# TRAIN_ROOT = "/kaggle/input/cv-dataset/Project Data/Food/Train"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# EPOCHS = 20
# STEPS_PER_EPOCH = 200
# IMG_SIZE = 224
# EMB_DIM = 256
# LR = 1e-4
# INITIAL_MARGIN = 0.2
# MAX_MARGIN = 0.35
# NEG_PER_CLASS = 2

# print("Device:", DEVICE)

# # ======================
# # DATA INDEX
# # ======================
# def build_index(root):
#     data = defaultdict(list)
#     for cls in os.listdir(root):
#         cls_path = os.path.join(root, cls)
#         if os.path.isdir(cls_path):
#             for img in os.listdir(cls_path):
#                 if img.lower().endswith((".jpg", ".png", ".jpeg")):
#                     data[cls].append(os.path.join(cls_path, img))
#     return data

# train_data = build_index(TRAIN_ROOT)
# classes = list(train_data.keys())
# print("Classes:", len(classes))

# # ======================
# # TRANSFORMS (CLIP)
# # ======================
# CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
# CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

# tf = transforms.Compose([
#     transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
#     transforms.RandomHorizontalFlip(),
#     transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
#     transforms.ToTensor(),
#     transforms.Normalize(CLIP_MEAN, CLIP_STD)
# ])

# def load_img(path):
#     return tf(Image.open(path).convert("RGB")).unsqueeze(0)

# # ======================
# # MODEL
# # ======================
# class SiameseModel(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.backbone = CLIPVisionModel.from_pretrained(
#             "openai/clip-vit-base-patch32"
#         )
#         self.projector = nn.Sequential(
#             nn.Linear(self.backbone.config.hidden_size, 512),
#             nn.LayerNorm(512),          # ✅ stable with batch=1
#             nn.GELU(),
#             nn.Linear(512, EMB_DIM)
#         )

#     def forward(self, x):
#         feat = self.backbone(pixel_values=x).last_hidden_state[:, 0, :]
#         emb = self.projector(feat)
#         return F.normalize(emb, dim=-1)

# model = SiameseModel().to(DEVICE)

# optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
# scaler = GradScaler()

# # ======================
# # SEMI-HARD TRIPLET SAMPLER
# # ======================
# @torch.no_grad()
# def sample_triplet(margin):
#     cls = random.choice(classes)
#     a_path, p_path = random.sample(train_data[cls], 2)

#     a = load_img(a_path).to(DEVICE)
#     p = load_img(p_path).to(DEVICE)

#     a_emb = model(a)
#     p_emb = model(p)
#     pos_dist = 1 - F.cosine_similarity(a_emb, p_emb)

#     neg_candidates = []
#     for c in classes:
#         if c != cls:
#             neg_candidates += random.sample(
#                 train_data[c], min(NEG_PER_CLASS, len(train_data[c]))
#             )

#     neg_imgs = torch.cat([load_img(p) for p in neg_candidates]).to(DEVICE)
#     neg_embs = model(neg_imgs)
#     neg_dists = 1 - F.cosine_similarity(a_emb, neg_embs)

#     mask = (neg_dists > pos_dist) & (neg_dists < pos_dist + margin)
#     idx = neg_dists[mask].argmin() if mask.any() else neg_dists.argmin()

#     return a_path, p_path, neg_candidates[idx.item()]

# # ======================
# # TRAINING LOOP
# # ======================
# print("\nStarting training...\n")

# margin = INITIAL_MARGIN

# for epoch in range(EPOCHS):
#     model.train()
#     epoch_loss = 0

#     margin = min(MAX_MARGIN, INITIAL_MARGIN + epoch * 0.01)

#     for step in range(STEPS_PER_EPOCH):
#         a_path, p_path, n_path = sample_triplet(margin)

#         a = load_img(a_path).to(DEVICE)
#         p = load_img(p_path).to(DEVICE)
#         n = load_img(n_path).to(DEVICE)

#         with autocast():
#             a_emb = model(a)
#             p_emb = model(p)
#             n_emb = model(n)

#             pos = 1 - F.cosine_similarity(a_emb, p_emb)
#             neg = 1 - F.cosine_similarity(a_emb, n_emb)
#             loss = F.relu(pos - neg + margin).mean()

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()

#         epoch_loss += loss.item()

#         if (step + 1) % 50 == 0:
#             print(
#                 f"Epoch [{epoch+1}/{EPOCHS}] "
#                 f"Step [{step+1}/{STEPS_PER_EPOCH}] "
#                 f"Loss: {loss.item():.4f} "
#                 f"Margin: {margin:.2f}"
#             )

#     print(
#         f"\nEpoch {epoch+1} DONE — Avg Loss: {epoch_loss / STEPS_PER_EPOCH:.4f}\n"
#     )

# # ======================
# # SAVE
# # ======================
# torch.save(model.state_dict(), "siamese_clip_strong.pth")
# print("✅ Model saved: siamese_clip_strong.pth")


In [3]:
# # ============================================================
# # COLLAPSE-PROOF SIAMESE NETWORK (CLIP BACKBONE)
# # Triplet Loss + Batch Mining + Variance Regularization
# # ============================================================

# import os, random
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torchvision import transforms
# from transformers import CLIPVisionModel
# from PIL import Image
# from collections import defaultdict
# from torch.cuda.amp import autocast, GradScaler

# # ======================
# # CONFIG
# # ======================
# TRAIN_ROOT = "/kaggle/input/cv-dataset/Project Data/Food/Train"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# EPOCHS = 20
# STEPS_PER_EPOCH = 200
# BATCH_SIZE = 8                 # ✅ IMPORTANT
# IMG_SIZE = 224
# EMB_DIM = 256
# LR = 3e-4                      # projector only
# MARGIN = 0.2                   # ✅ FIXED
# NEG_PER_CLASS = 2
# VAR_WEIGHT = 0.1               # anti-collapse

# print("Device:", DEVICE)

# # ======================
# # DATA INDEX
# # ======================
# def build_index(root):
#     data = defaultdict(list)
#     for cls in os.listdir(root):
#         cls_path = os.path.join(root, cls)
#         if os.path.isdir(cls_path):
#             for img in os.listdir(cls_path):
#                 if img.lower().endswith((".jpg", ".png", ".jpeg")):
#                     data[cls].append(os.path.join(cls_path, img))
#     return data

# train_data = build_index(TRAIN_ROOT)
# classes = list(train_data.keys())
# print("Classes:", len(classes))

# # ======================
# # TRANSFORMS
# # ======================
# CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
# CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

# tf = transforms.Compose([
#     transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
#     transforms.RandomHorizontalFlip(),
#     transforms.ColorJitter(0.2,0.2,0.2,0.05),
#     transforms.ToTensor(),
#     transforms.Normalize(CLIP_MEAN, CLIP_STD)
# ])

# def load_img(path):
#     return tf(Image.open(path).convert("RGB"))

# # ======================
# # MODEL
# # ======================
# class SiameseModel(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.backbone = CLIPVisionModel.from_pretrained(
#             "openai/clip-vit-base-patch32"
#         )
#         self.projector = nn.Sequential(
#             nn.Linear(self.backbone.config.hidden_size, 512),
#             nn.LayerNorm(512),
#             nn.GELU(),
#             nn.Linear(512, EMB_DIM)
#         )

#     def forward(self, x):
#         feat = self.backbone(pixel_values=x).last_hidden_state[:, 0, :]
#         emb = self.projector(feat)
#         return F.normalize(emb, dim=-1)

# model = SiameseModel().to(DEVICE)

# # ✅ FREEZE BACKBONE (CRITICAL)
# for p in model.backbone.parameters():
#     p.requires_grad = False

# optimizer = torch.optim.AdamW(
#     model.projector.parameters(), lr=LR, weight_decay=1e-4
# )
# scaler = GradScaler()

# # ======================
# # VARIANCE LOSS (ANTI-COLLAPSE)
# # ======================
# def variance_loss(emb, eps=1e-4):
#     std = torch.sqrt(emb.var(dim=0) + eps)
#     return torch.mean(F.relu(1 - std))

# # ======================
# # BATCH TRIPLET SAMPLER
# # ======================
# def sample_batch():
#     A, P, N = [], [], []

#     for _ in range(BATCH_SIZE):
#         cls = random.choice(classes)
#         a, p = random.sample(train_data[cls], 2)

#         neg_cls = random.choice([c for c in classes if c != cls])
#         n = random.choice(train_data[neg_cls])

#         A.append(load_img(a))
#         P.append(load_img(p))
#         N.append(load_img(n))

#     return (
#         torch.stack(A).to(DEVICE),
#         torch.stack(P).to(DEVICE),
#         torch.stack(N).to(DEVICE)
#     )

# # ======================
# # TRAINING LOOP
# # ======================
# print("\nStarting COLLAPSE-PROOF training...\n")

# for epoch in range(EPOCHS):
#     model.train()
#     epoch_loss = 0

#     for step in range(STEPS_PER_EPOCH):
#         a, p, n = sample_batch()

#         with autocast():
#             a_emb = model(a)
#             p_emb = model(p)
#             n_emb = model(n)

#             pos = 1 - F.cosine_similarity(a_emb, p_emb)
#             neg = 1 - F.cosine_similarity(a_emb, n_emb)

#             triplet_loss = F.relu(pos - neg + MARGIN).mean()
#             var_loss = variance_loss(
#                 torch.cat([a_emb, p_emb, n_emb], dim=0)
#             )

#             loss = triplet_loss + VAR_WEIGHT * var_loss

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()

#         epoch_loss += loss.item()

#         if (step + 1) % 50 == 0:
#             print(
#                 f"Epoch [{epoch+1}/{EPOCHS}] "
#                 f"Step [{step+1}/{STEPS_PER_EPOCH}] "
#                 f"Loss: {loss.item():.4f} "
#                 f"Triplet: {triplet_loss.item():.4f} "
#                 f"Var: {var_loss.item():.4f}"
#             )

#     print(
#         f"\nEpoch {epoch+1} DONE — Avg Loss: {epoch_loss / STEPS_PER_EPOCH:.4f}\n"
#     )

# # ======================
# # SAVE
# # ======================
# torch.save(model.state_dict(), "siamese_clip_NO_COLLAPSE.pth")
# print("✅ Saved: siamese_clip_NO_COLLAPSE.pth")


In [4]:
# ============================================================
# ENHANCED SIAMESE NETWORK (CLIP BACKBONE)
# Triplet Loss + Batch Mining + Variance Regularization + Advanced Training
# ============================================================

import os, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from transformers import CLIPVisionModel
from PIL import Image
from collections import defaultdict
from torch.cuda.amp import autocast, GradScaler

# ======================
# CONFIG
# ======================
TRAIN_ROOT = "/kaggle/input/cv-dataset/Project Data/Food/Train"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 30               # more epochs for better embedding convergence
STEPS_PER_EPOCH = 250     # more steps per epoch
BATCH_SIZE = 16           # larger batch to stabilize training
IMG_SIZE = 224
EMB_DIM = 256
LR = 3e-4
MARGIN = 0.25             # slightly larger margin
NEG_PER_CLASS = 3
VAR_WEIGHT = 0.1           # anti-collapse
DROPOUT = 0.1              # projector dropout for regularization

print("Device:", DEVICE)

# ======================
# DATA INDEX
# ======================
def build_index(root):
    data = defaultdict(list)
    for cls in os.listdir(root):
        cls_path = os.path.join(root, cls)
        if os.path.isdir(cls_path):
            for img in os.listdir(cls_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    data[cls].append(os.path.join(cls_path, img))
    return data

train_data = build_index(TRAIN_ROOT)
classes = list(train_data.keys())
print("Classes:", len(classes))

# ======================
# TRANSFORMS (ADVANCED)
# ======================
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.RandomRotation(15),           # rotation augmentation
    transforms.RandomAffine(0, translate=(0.1,0.1)),  # translation
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD)
])

def load_img(path):
    return tf(Image.open(path).convert("RGB"))

# ======================
# MODEL
# ======================
class EnhancedSiamese(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        self.projector = nn.Sequential(
            nn.Linear(self.backbone.config.hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(512, EMB_DIM)
        )

    def forward(self, x):
        feat = self.backbone(pixel_values=x).last_hidden_state[:, 0, :]
        emb = self.projector(feat)
        return F.normalize(emb, dim=-1)

model = EnhancedSiamese().to(DEVICE)

# ✅ Freeze backbone to prevent overfitting / collapse
for p in model.backbone.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(model.projector.parameters(), lr=LR, weight_decay=1e-4)
scaler = GradScaler()

# ======================
# VARIANCE LOSS (ANTI-COLLAPSE)
# ======================
def variance_loss(emb, eps=1e-4):
    std = torch.sqrt(emb.var(dim=0) + eps)
    return torch.mean(F.relu(1 - std))

# ======================
# BATCH TRIPLET SAMPLER
# ======================
def sample_batch():
    A, P, N = [], [], []
    for _ in range(BATCH_SIZE):
        cls = random.choice(classes)
        a, p = random.sample(train_data[cls], 2)
        neg_cls = random.choice([c for c in classes if c != cls])
        n = random.choice(train_data[neg_cls])

        A.append(load_img(a))
        P.append(load_img(p))
        N.append(load_img(n))

    return torch.stack(A).to(DEVICE), torch.stack(P).to(DEVICE), torch.stack(N).to(DEVICE)

# ======================
# TRAINING LOOP (ENHANCED)
# ======================
print("\nStarting enhanced Siamese training...\n")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for step in range(STEPS_PER_EPOCH):
        a, p, n = sample_batch()

        with autocast():
            a_emb = model(a)
            p_emb = model(p)
            n_emb = model(n)

            pos = 1 - F.cosine_similarity(a_emb, p_emb)
            neg = 1 - F.cosine_similarity(a_emb, n_emb)

            triplet_loss = F.relu(pos - neg + MARGIN).mean()
            var_loss = variance_loss(torch.cat([a_emb, p_emb, n_emb], dim=0))

            # **Add L2 regularization to embeddings**
            l2_loss = 0.01 * (a_emb.norm(dim=1).mean() + p_emb.norm(dim=1).mean() + n_emb.norm(dim=1).mean())

            loss = triplet_loss + VAR_WEIGHT * var_loss + l2_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(
                f"Epoch [{epoch+1}/{EPOCHS}] Step [{step+1}/{STEPS_PER_EPOCH}] "
                f"Loss: {loss.item():.4f} Triplet: {triplet_loss.item():.4f} Var: {var_loss.item():.4f}"
            )

    print(f"\nEpoch {epoch+1} DONE — Avg Loss: {epoch_loss / STEPS_PER_EPOCH:.4f}\n")

# ======================
# SAVE MODEL
# ======================
torch.save(model.state_dict(), "enhanced_siamese_clip.pth")
print("✅ Enhanced Siamese model saved: enhanced_siamese_clip.pth")


2025-12-19 23:05:34.553769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766185534.767950      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766185534.835031      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766185535.357323      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766185535.357358      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766185535.357361      23 computation_placer.cc:177] computation placer alr

Device: cuda
Classes: 85


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/tmp/ipykernel_23/4208968960.py:98: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Starting enhanced Siamese training...



/tmp/ipykernel_23/4208968960.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/30] Step [50/250] Loss: 0.2501 Triplet: 0.1258 Var: 0.9434
Epoch [1/30] Step [100/250] Loss: 0.2225 Triplet: 0.0982 Var: 0.9434
Epoch [1/30] Step [150/250] Loss: 0.2136 Triplet: 0.0894 Var: 0.9421
Epoch [1/30] Step [200/250] Loss: 0.1411 Triplet: 0.0168 Var: 0.9433
Epoch [1/30] Step [250/250] Loss: 0.1873 Triplet: 0.0631 Var: 0.9424

Epoch 1 DONE — Avg Loss: 0.1949

Epoch [2/30] Step [50/250] Loss: 0.2792 Triplet: 0.1549 Var: 0.9427
Epoch [2/30] Step [100/250] Loss: 0.1538 Triplet: 0.0293 Var: 0.9455
Epoch [2/30] Step [150/250] Loss: 0.2501 Triplet: 0.1259 Var: 0.9418
Epoch [2/30] Step [200/250] Loss: 0.1326 Triplet: 0.0083 Var: 0.9428
Epoch [2/30] Step [250/250] Loss: 0.1705 Triplet: 0.0465 Var: 0.9403

Epoch 2 DONE — Avg Loss: 0.1769

Epoch [3/30] Step [50/250] Loss: 0.1475 Triplet: 0.0234 Var: 0.9410
Epoch [3/30] Step [100/250] Loss: 0.1853 Triplet: 0.0610 Var: 0.9429
Epoch [3/30] Step [150/250] Loss: 0.1407 Triplet: 0.0165 Var: 0.9420
Epoch [3/30] Step [200/250] Loss: 0.22